<a href="https://colab.research.google.com/github/kartik2006-del/biopapers-rag/blob/main/BioPapers_RAG_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cleaning


In [1]:

import requests
import pandas as pd
import xml.etree.ElementTree as ET




In [2]:
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db": "pubmed",
    "term": '"CRISPR"[Title/Abstract] AND "gene editing"[Title/Abstract]',
    "retmax": 5,
    "retmode": "json"
}

response = requests.get(url, params=params, timeout=30)

print("Status code:", response.status_code)
print("PMIDs:", response.json()["esearchresult"]["idlist"])

Status code: 200
PMIDs: ['42929859', '42928503', '42926548', '42926261', '42926223']


In [3]:
fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

fetch_params = {
    "db": "pubmed",
    "id": ",".join(response.json()["esearchresult"]["idlist"]),
    "rettype": "abstract",
    "retmode": "xml"
}

fetch_response = requests.get(
    fetch_url,
    params=fetch_params,
    timeout=30
)

print("Status code:", fetch_response.status_code)
print(fetch_response.text[:1000])

Status code: 200
<?xml version="1.0" ?>
<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January 2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">
<PubmedArticleSet>
<PubmedArticle><MedlineCitation Status="Publisher" Owner="NLM"><PMID Version="1">42929859</PMID><DateRevised><Year>2026</Year><Month>10</Month><Day>26</Day></DateRevised><Article PubModel="Electronic"><Journal><ISSN IssnType="Electronic">2073-4409</ISSN><JournalIssue CitedMedium="Internet"><Volume>15</Volume><Issue>20</Issue><PubDate><Year>2026</Year><Month>Oct</Month><Day>16</Day></PubDate></JournalIssue><Title>Cells</Title><ISOAbbreviation>Cells</ISOAbbreviation></Journal><ArticleTitle>Artificial Intelligence as a Catalyst for Cell-Based Therapies in Retinal Disease.</ArticleTitle><ELocationID EIdType="pii" ValidYN="Y">1887</ELocationID><ELocationID EIdType="doi" ValidYN="Y">10.3390/cells15201887</ELocationID><Abstract><AbstractText>Retinal degenerative diseases, including age-relate

In [4]:
root = ET.fromstring(fetch_response.content)

article = root.find(".//PubmedArticle")

title = article.findtext(
    ".//ArticleTitle",
    default="Title not found"
)

print("First paper title:")
print(title)

First paper title:
Artificial Intelligence as a Catalyst for Cell-Based Therapies in Retinal Disease.


In [5]:
articles = root.findall(".//PubmedArticle")

for article in articles:
    pmid = article.findtext(".//PMID", default="N/A")
    title = article.findtext(".//ArticleTitle", default="N/A")

    print("PMID:", pmid)
    print("Title:", title)
    print("-" * 80)

PMID: 42929859
Title: Artificial Intelligence as a Catalyst for Cell-Based Therapies in Retinal Disease.
--------------------------------------------------------------------------------
PMID: 42928503
Title: In Vivo CRISPR Screen Identifies RPS19 as an Immunotherapy Target in Hepatocellular Carcinoma.
--------------------------------------------------------------------------------
PMID: 42926548
Title: p38α MAPK integrates opposing oncogenic APC/β-catenin/TCF and GUCY2C tumor-suppressor axes in colorectal cancer.
--------------------------------------------------------------------------------
PMID: 42926261
Title: Generation of Functional Gene Knockout Melanoma Cell Lines by CRISPR-Cas9 Gene Editing.
--------------------------------------------------------------------------------
PMID: 42926223
Title: CAR-NK cell immunotherapy: a comprehensive review of engineering platforms, clinical applications in hematological and solid tumors.
------------------------------------------------------

In [6]:

records = []

for article in articles:
    pmid = article.findtext(".//PMID", default="N/A")
    title = article.findtext(".//ArticleTitle", default="N/A")

    journal = article.findtext(
        ".//Journal/Title",
        default="N/A"
    )

    abstract_parts = [
        part.itertext()
        for part in article.findall(".//Abstract/AbstractText")
    ]

    abstract = " ".join(
        text
        for part in abstract_parts
        for text in part
    )

    records.append({
        "PMID": pmid,
        "Title": title,
        "Journal": journal,
        "Abstract": abstract,
        "Source_URL": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
    })

print("Records parsed:", len(records))
print("First record:", records[0])


Records parsed: 5
First record: {'PMID': '42929859', 'Title': 'Artificial Intelligence as a Catalyst for Cell-Based Therapies in Retinal Disease.', 'Journal': 'Cells', 'Abstract': 'Retinal degenerative diseases, including age-related macular degeneration (AMD), retinitis pigmentosa (RP), and Stargardt disease, represent leading causes of irreversible vision loss worldwide, collectively affecting hundreds of millions of people. Cell-based therapies, particularly those leveraging induced pluripotent stem cells (iPSCs), retinal pigment epithelium (RPE) transplantation, and CRISPR-Cas9 gene editing, have emerged as promising treatment modalities that may preserve, replace, or partially restore retinal function. However, these approaches face significant challenges in patient stratification, manufacturing consistency, surgical precision, and long-term outcome monitoring. Artificial intelligence (AI), encompassing machine learning (ML), deep learning (DL), and computer vision, is increasingl

In [7]:
df = pd.DataFrame(records)

print("Dataset shape:", df.shape)
display(df[["PMID", "Title", "Journal"]])

Dataset shape: (5, 5)


,PMID,Title,Journal
0,42929859,Artificial Intelligence as a Catalyst for Cell...,Cells
1,42928503,In Vivo CRISPR Screen Identifies RPS19 as an I...,"Advanced science (Weinheim, Baden-Wurttemberg,..."
2,42926548,p38α MAPK integrates opposing oncogenic APC/β-...,bioRxiv : the preprint server for biology
3,42926261,Generation of Functional Gene Knockout Melanom...,"Methods in molecular biology (Clifton, N.J.)"
4,42926223,CAR-NK cell immunotherapy: a comprehensive rev...,Journal of cancer research and clinical oncology


In [8]:

for article in articles:
    print("PMID:", article.findtext(".//PMID", default="N/A"))

    authors = []
    for author in article.findall("./MedlineCitation/Article/AuthorList/Author"):
        fore = author.findtext("ForeName", default="")
        last = author.findtext("LastName", default="")
        name = f"{fore} {last}".strip()

        if name:
            authors.append(name)

    print("Authors:", ", ".join(authors) if authors else "N/A")

    pub_date = article.find("./MedlineCitation/Article/Journal/JournalIssue/PubDate")

    year = "N/A"
    if pub_date is not None:
        year = pub_date.findtext("Year", default="N/A")

        if year == "N/A":
            medline_date = pub_date.findtext("MedlineDate", default="")
            if medline_date:
                year = medline_date[:4]

    print("Publication year:", year)
    print("-" * 60)


PMID: 42929859
Authors: Girish K Srivastava, Priyank Shukla, Sara Crespo Millas, Yogeswaran Lokanathan
Publication year: 2026
------------------------------------------------------------
PMID: 42928503
Authors: Lei He, Duo Yu, Fei Xie, Kaisheng Hou, Zhaozhao Li, Meifang Dai, Jingyi Yun, Haiyan Cao, Yang Xiao, Yueyuan Qiu, Gege Shi, Wangqian Zhang, Shuning Wang, Qiang Hao, Jieyu Zhang, Zhaowei Wang, Yingqi Zhang, Kuo Zhang, Jintao Gu, Haichen Du, Meng Li
Publication year: 2026
------------------------------------------------------------
PMID: 42926548
Authors: Ariana A Entezari, Adi Caspi, Jasmine R Alvarez, Andrew E Evans, Jeffrey A Rappaport, Thomas J M Kuret, Allison S Doermann, Ramkrishna Mitra, Adam E Snook, Scott A Waldman
Publication year: 2026
------------------------------------------------------------
PMID: 42926261
Authors: Kristian M Hargadon, David Z Bushhouse, Coleman E Johnson, Corey J Willams
Publication year: 2027
--------------------------------------------------------

In [9]:

records = []

for article in articles:
    pmid = article.findtext(".//PMID", default="N/A")
    title = article.findtext(".//ArticleTitle", default="N/A")
    journal = article.findtext(
        "./MedlineCitation/Article/Journal/Title",
        default="N/A"
    )

    abstract_parts = [
        "".join(part.itertext())
        for part in article.findall(
            "./MedlineCitation/Article/Abstract/AbstractText"
        )
    ]
    abstract = " ".join(abstract_parts)

    pub_date = article.find(
        "./MedlineCitation/Article/Journal/JournalIssue/PubDate"
    )
    year = "N/A"

    if pub_date is not None:
        year = pub_date.findtext("Year", default="N/A")
        if year == "N/A":
            medline_date = pub_date.findtext("MedlineDate", default="")
            if medline_date:
                year = medline_date[:4]

    authors = []
    for author in article.findall(
        "./MedlineCitation/Article/AuthorList/Author"
    ):
        fore = author.findtext("ForeName", default="")
        last = author.findtext("LastName", default="")
        name = f"{fore} {last}".strip()
        if name:
            authors.append(name)

    records.append({
        "PMID": pmid,
        "Title": title,
        "Year": year,
        "Authors": ", ".join(authors),
        "Journal": journal,
        "Abstract": abstract,
        "Source_URL": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
    })

df = pd.DataFrame(records)

print("Dataset shape:", df.shape)
display(df[["PMID", "Year", "Title", "Authors", "Journal"]])


Dataset shape: (5, 7)


,PMID,Year,Title,Authors,Journal
0,42929859,2026,Artificial Intelligence as a Catalyst for Cell...,"Girish K Srivastava, Priyank Shukla, Sara Cres...",Cells
1,42928503,2026,In Vivo CRISPR Screen Identifies RPS19 as an I...,"Lei He, Duo Yu, Fei Xie, Kaisheng Hou, Zhaozha...","Advanced science (Weinheim, Baden-Wurttemberg,..."
2,42926548,2026,p38α MAPK integrates opposing oncogenic APC/β-...,"Ariana A Entezari, Adi Caspi, Jasmine R Alvare...",bioRxiv : the preprint server for biology
3,42926261,2027,Generation of Functional Gene Knockout Melanom...,"Kristian M Hargadon, David Z Bushhouse, Colema...","Methods in molecular biology (Clifton, N.J.)"
4,42926223,2026,CAR-NK cell immunotherapy: a comprehensive rev...,"Salma Mohamed Awadallah, NourEldin Elmesalamy,...",Journal of cancer research and clinical oncology


In [16]:
for col in df.columns:
    print(f"\nColumn: {col}")
    print("Missing values:", df[col].isnull().sum())
    print("Blank values:", df[col].astype(str).str.strip().eq("").sum())
    print("Duplicate values:", df[col].duplicated().sum())


Column: PMID
Missing values: 0
Blank values: 0
Duplicate values: 0

Column: Title
Missing values: 0
Blank values: 0
Duplicate values: 0

Column: Year
Missing values: 0
Blank values: 0
Duplicate values: 3

Column: Authors
Missing values: 0
Blank values: 0
Duplicate values: 0

Column: Journal
Missing values: 0
Blank values: 0
Duplicate values: 0

Column: Abstract
Missing values: 0
Blank values: 0
Duplicate values: 0

Column: Source_URL
Missing values: 0
Blank values: 0
Duplicate values: 0


In [19]:
for i, year in enumerate(df["Year"]):
    print(f"Row {i}: Year = {year}")

Row 0: Year = 2026
Row 1: Year = 2026
Row 2: Year = 2026
Row 3: Year = 2027
Row 4: Year = 2026


In [20]:
for i, row in df.iterrows():
    if int(row["Year"]) > 2026:
        print("PMID:", row["PMID"])
        print("Title:", row["Title"])
        print("Year:", row["Year"])
        print("Source:", row["Source_URL"])

PMID: 42926261
Title: Generation of Functional Gene Knockout Melanoma Cell Lines by CRISPR-Cas9 Gene Editing.
Year: 2027
Source: https://pubmed.ncbi.nlm.nih.gov/42926261/


In [26]:
df.to_csv("pubmed_papers.csv", index=False)